# 6. Interactive Visual Analytics with Folium
Marks launch sites, shows success/failure per launch with marker clusters, and measures distance from a site to nearby features (coast, railway, highway, city).

In [ ]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby('Launch Site', as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

## Task 1 - mark all launch sites

In [ ]:
nasa = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa, zoom_start=5)
for _, r in launch_sites_df.iterrows():
    coord = [r['Lat'], r['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html=f'<div style="font-size:12; color:#d35400;"><b>{r["Launch Site"]}</b></div>')).add_to(site_map)
site_map

## Task 2 - success/failed launches per site (green = success, red = failure)

In [ ]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)
for _, r in spacex_df.iterrows():
    marker_cluster.add_child(folium.Marker([r['Lat'], r['Long']],
        icon=folium.Icon(color='white', icon_color=r['marker_color'])))
site_map

## Task 3 - distances to proximities
Use the mouse-position readout to pick coordinates of the nearest coastline point, then compute the distance.

**Replace the example coordinates with points you pick on your own map.**

In [ ]:
site_map.add_child(MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
                                 num_digits=20, prefix='Lat:'))
site_map

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    dlon, dlat = radians(lon2 - lon1), radians(lat2 - lat1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# CCAFS SLC-40 and an example coastline point (EDIT with the point you picked on the map)
launch_site_lat, launch_site_lon = 28.563197, -80.576820
coast_lat, coast_lon = 28.56367, -80.56795
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coast_lat, coast_lon)
print(f'Distance to coastline: {distance_coastline:.2f} km')

In [ ]:
folium.Marker([coast_lat, coast_lon],
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
    html=f'<div style="font-size:12; color:#252526;"><b>{distance_coastline:.2f} KM</b></div>')).add_to(site_map)
folium.PolyLine([[launch_site_lat, launch_site_lon], [coast_lat, coast_lon]], weight=1).add_to(site_map)
site_map

Repeat the last two cells for the nearest railway, highway and city (pick the coordinates with the mouse readout).

In [ ]:
site_map.save('launch_site_map.html')   # keep this HTML file in the repo as well